# cognitive_alpha_evolution_full_v1

Full CogAlpha goal-runner profile for SOXL. This is research evidence, not a production trading workflow.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qlib.data import D
from qlib.workflow import R
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP

from src.data import QlibProviderConfig, alpha158_legacy_feature_config, build_alpha158_legacy_dataset, init_qlib_provider
from src.factor import CognitiveAlphaCandidate, cognitive_alpha_codex_prompt_seed
from src.workflow import (
    CognitiveAlphaEvolutionSpec,
    build_cognitive_alpha_code_candidate_provider,
    build_cognitive_alpha_evolution_profile,
    run_cognitive_alpha_evolution,
)

CARD = ROOT / "research" / "cognitive_alpha_evolution_full_v1"
EXPERIMENT = "cognitive_alpha_evolution_full_v1"
PROVIDER_URI = ROOT / "data" / "qlib" / "soxl"
RECORDER_URI = CARD / "qlib_recorder"
MANIFEST = json.loads((PROVIDER_URI / "qlib_store_manifest.json").read_text(encoding="utf-8"))
AS_OF = MANIFEST["calendar_end"]
DATA_START = "2019-01-02"
TRAIN_END = "2025-12-31"
TEST_START = "2026-01-02"
MARKET = "all"
REGION = "us"
INSTRUMENT = "SOXL"

CARD.mkdir(parents=True, exist_ok=True)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
print(json.dumps({"experiment": EXPERIMENT, "as_of": AS_OF, "provider_uri": PROVIDER_URI.as_posix()}, indent=2))

{
  "experiment": "cognitive_alpha_evolution_full_v1",
  "as_of": "2026-06-24",
  "provider_uri": "/Users/rziiiii/dev/workspace/data/qlib/soxl"
}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## qlib data and Alpha158

In [2]:
config = QlibProviderConfig(provider_uri=PROVIDER_URI, region=REGION, market=MARKET)
provider_meta = init_qlib_provider(config)
raw = D.features(
    [INSTRUMENT],
    ["$open", "$high", "$low", "$close", "$volume", "$vix"],
    start_time=DATA_START,
    end_time=AS_OF,
    freq="day",
)
base = raw.reset_index().rename(
    columns={"$open": "open", "$high": "high", "$low": "low", "$close": "close", "$volume": "volume", "$vix": "vix"}
)
base = base.loc[base["instrument"] == INSTRUMENT].sort_values("datetime").reset_index(drop=True)
base["datetime"] = pd.to_datetime(base["datetime"])
base["label_return_5d"] = base["close"].pct_change(5).shift(-5).ffill().fillna(0.0)

# Alpha158.get_feature_config is accessed through the project wrapper for this qlib version.
alpha158_fields, alpha158_names, alpha158_config = alpha158_legacy_feature_config()
segments = {"train": (DATA_START, TRAIN_END), "test": (TEST_START, AS_OF)}
alpha158_dataset = build_alpha158_legacy_dataset(
    config,
    instruments=[INSTRUMENT],
    label="Ref($close, -5) / $close - 1",
    train=(DATA_START, TRAIN_END),
    test=(TEST_START, AS_OF),
)
assert isinstance(alpha158_dataset, DatasetH)
assert isinstance(alpha158_dataset.handler, DataHandlerLP)
alpha158_frame = pd.concat([
    alpha158_dataset.prepare("train", col_set="feature", data_key=DataHandlerLP.DK_L),
    alpha158_dataset.prepare("test", col_set="feature", data_key=DataHandlerLP.DK_L),
]).reset_index()
alpha158_frame = alpha158_frame.loc[alpha158_frame["instrument"] == INSTRUMENT].copy()
alpha158_frame["datetime"] = pd.to_datetime(alpha158_frame["datetime"])
alpha158_cols = [col for col in alpha158_frame.columns if col not in {"instrument", "datetime"}]
selected_alpha158_cols = alpha158_cols[:72]
alpha_prefix_map = {col: f"alpha158_{col}" for col in selected_alpha158_cols}
ml_frame = base.merge(
    alpha158_frame[["datetime", *selected_alpha158_cols]].rename(columns=alpha_prefix_map),
    on="datetime",
    how="left",
)
alpha_candidate_columns = list(alpha_prefix_map.values())
print(json.dumps({"rows": len(ml_frame), "alpha158_total": len(alpha158_cols), "alpha158_candidates": len(alpha_candidate_columns)}, indent=2))

[33754:MainThread](2026-06-25 18:56:56,801) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[33754:MainThread](2026-06-25 18:56:56,804) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[33754:MainThread](2026-06-25 18:56:56,804) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[33754:MainThread](2026-06-25 18:56:56,817) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[33754:MainThread](2026-06-25 18:56:56,818) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[33754:MainThread](2026-06-25 18:56:56,818) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[33754:MainThread](2026-06-25 18:56:56,879) INFO - qlib.timer - [log.py:127] - Time cost: 0.057s | Loading data Done


[33754:MainThread](2026-06-25 18:56:56,879) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[33754:MainThread](2026-06-25 18:56:56,880) INFO - qlib.timer - [log.py:127] - Time cost: 0.058s | Init data Done


{
  "rows": 1879,
  "alpha158_total": 166,
  "alpha158_candidates": 72
}


## Full CogAlpha evolution

In [3]:
profile = build_cognitive_alpha_evolution_profile("soxl_full")
assert profile.initial_pool_size == 80
assert profile.parent_pool_size == 32
assert profile.children_per_generation == 96
assert profile.generations == 24

leaky_candidate = CognitiveAlphaCandidate(
    name="factor_future_leak_full_demo",
    level="geometric_fusion",
    agent="AgentCreative",
    theme="future leakage repair full test",
    required_columns=("close",),
    lookback=1,
    declared_lookahead=1,
    code="df['close'].shift(-1) / df['close'] - 1",
    economic_rationale="Invalid future leak included only to prove the quality checker and repair path.",
    function=lambda data: data["close"].shift(-1) / data["close"] - 1.0,
)
candidate_provider = build_cognitive_alpha_code_candidate_provider()
initial_candidate_count = profile.initial_pool_size + 1

qlib_workflow_api = [
    "qlib.init",
    "qlib.data.D.features",
    "DatasetH",
    "DataHandlerLP",
    "Alpha158.get_feature_config",
    "qlib.workflow.R.start",
]

runtime_tmp = Path(tempfile.mkdtemp(prefix="cogalpha_full_"))
try:
    R.set_uri(RECORDER_URI.as_posix())
    with R.start(experiment_name=EXPERIMENT, recorder_name="cognitive_alpha_evolution_full"):
        recorder = R.get_recorder()
        spec = CognitiveAlphaEvolutionSpec(
            run_id="cognitive_alpha_evolution",
            profile=profile.profile,
            runtime_dir=runtime_tmp,
            generations=profile.generations,
            parent_pool_size=profile.parent_pool_size,
            children_per_generation=profile.children_per_generation,
            initial_pool_size=profile.initial_pool_size,
            qualified_percentile=0.65,
            elite_percentile=0.80,
            max_abs_correlation=0.95,
            min_observations=180,
            max_missing_ratio=0.55,
            min_distinct_values=5,
            fallback_top_n=8,
        )
        evolution = run_cognitive_alpha_evolution(
            ml_frame,
            spec=spec,
            label_col="label_return_5d",
            initial_candidates=[leaky_candidate],
            candidate_provider=candidate_provider,
        )
        for state_name in ["state.json", "summary.json"]:
            state_path = evolution.run_dir / state_name
            payload = json.loads(state_path.read_text(encoding="utf-8"))
            payload["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
            if isinstance(payload.get("spec"), dict):
                payload["spec"]["runtime_dir"] = "recorder:artifacts"
            state_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
        recorder.log_params(
            provider_uri=PROVIDER_URI.as_posix(),
            profile=profile.profile,
            initial_pool_size=profile.initial_pool_size,
            parent_pool_size=profile.parent_pool_size,
            children_per_generation=profile.children_per_generation,
            generations=profile.generations,
            candidate_provider=candidate_provider.provider_id,
            train_start=DATA_START,
            train_end=TRAIN_END,
            test_start=TEST_START,
            test_end=AS_OF,
        )
        recorder.log_metrics(
            selected_count=float(evolution.summary["selected_count"]),
            candidate_metric_rows=float(evolution.summary["candidate_metric_rows"]),
            repair_attempt_count=float(evolution.summary["repair_attempt_count"]),
        )
        for path in evolution.run_dir.rglob("*"):
            if path.is_file():
                rel_parent = path.parent.relative_to(evolution.run_dir).as_posix()
                artifact_path = "cognitive_alpha_evolution" if rel_parent == "." else f"cognitive_alpha_evolution/{rel_parent}"
                recorder.log_artifact(path.as_posix(), artifact_path=artifact_path)
        recorder_id = recorder.id
        recorder_local_dir = Path(recorder.get_local_dir()).resolve()
finally:
    shutil.rmtree(runtime_tmp, ignore_errors=True)

metrics = evolution.generation_metrics.copy()
max_candidate_name_length = int(metrics["factor"].astype(str).str.len().max())
print(json.dumps({"recorder_id": recorder_id, "selected_count": evolution.summary["selected_count"], "rows": len(metrics), "max_candidate_name_length": max_candidate_name_length}, indent=2))

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[33754:MainThread](2026-06-25 18:56:56,900) INFO - qlib.workflow - [exp.py:258] - Experiment 473475477030232762 starts running ...


[33754:MainThread](2026-06-25 18:56:56,965) INFO - qlib.workflow - [recorder.py:345] - Recorder f2930a051cfc46b39a12f4fe26f417a4 starts running under Experiment 473475477030232762 ...


[33754:MainThread](2026-06-25 18:57:10,824) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "recorder_id": "f2930a051cfc46b39a12f4fe26f417a4",
  "selected_count": 1,
  "rows": 2289,
  "max_candidate_name_length": 88
}


## Card sidecars

In [4]:
selected_features_path = CARD / "selected_features.csv"
pd.DataFrame({"selected_feature": evolution.selected_feature_names}).to_csv(selected_features_path, index=False)

evolution_summary = dict(evolution.summary)
evolution_summary["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
evolution_summary["spec"]["runtime_dir"] = "recorder:artifacts"
evolution_summary["max_candidate_name_length"] = max_candidate_name_length
(CARD / "evolution_summary.json").write_text(json.dumps(evolution_summary, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "failure_modes.md").write_text(
    "# Failure Modes\n\n"
    "- This is the full CogAlpha evolution runner profile, not a promoted position strategy.\n"
    "- Candidate generation uses the repo-owned LLM/Code provider boundary (`deterministic_code_agent_pool`) with 21 CogAlpha agents and 5 guidance styles; external paid LLM providers are not required for this reproducible run.\n"
    "- Full evolution output is recorder-first; benchmarked position modeling remains in the downstream position card.\n"
    "- 本研究面向交易复盘和风险控制，不构成买卖建议。\n",
    encoding="utf-8",
)

workflow_yaml = "\n".join([
    f"experiment: {EXPERIMENT}",
    "card_type: diagnostic_cognitive_alpha_full_goal_runner",
    "paper_reference:",
    "  arxiv_id: '2511.18850'",
    "  method: checkpointed_cognitive_alpha_evolution_full",
    "qlib:",
    f"  provider_uri: {PROVIDER_URI.as_posix()}",
    f"  recorder_uri: {RECORDER_URI.as_posix()}",
    f"  market: {MARKET}",
    f"  region: {REGION}",
    "segments:",
    f"  train: [{DATA_START}, {TRAIN_END}]",
    f"  test: [{TEST_START}, {AS_OF}]",
    "evolution:",
    "  profile: soxl_full",
    f"  generations: {profile.generations}",
    f"  parent_pool_size: {profile.parent_pool_size}",
    f"  children_per_generation: {profile.children_per_generation}",
    "artifact_policy:",
    "  policy: slim_card_recorder_first",
    "  workflow_config: workflow.yaml",
    "  machine_entrypoint: summary.json",
    "",
])
(CARD / "workflow.yaml").write_text(workflow_yaml, encoding="utf-8")

for transient in ["code_cached.txt", "code_diff.txt", "code_status.txt"]:
    transient_path = recorder_local_dir / "artifacts" / transient
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(
    path.relative_to(recorder_local_dir / "artifacts").as_posix()
    for path in (recorder_local_dir / "artifacts").rglob("*")
    if path.is_file()
)
summary = {
    "experiment": EXPERIMENT,
    "card_type": "diagnostic_cognitive_alpha_full_goal_runner",
    "instrument": INSTRUMENT,
    "market": MARKET,
    "region": REGION,
    "as_of": AS_OF,
    "data_range": [DATA_START, AS_OF],
    "segments": {"train": [DATA_START, TRAIN_END], "test": [TEST_START, AS_OF]},
    "provider_uri": PROVIDER_URI.as_posix(),
    "recorder_uri": RECORDER_URI.as_posix(),
    "recorder_id": recorder_id,
    "recorder_local_dir": recorder_local_dir.as_posix(),
    "paper_reference": {"arxiv_id": "2511.18850", "method": "checkpointed_cognitive_alpha_evolution_full", "version_checked": "v3"},
    "prompt_seed": cognitive_alpha_codex_prompt_seed(),
    "evolution": evolution_summary,
    "promotion_decision": "not_promoted_full_evolution_only",
    "qlib_docs": {
        "data": "https://qlib.readthedocs.io/en/latest/component/data.html",
        "workflow": "https://qlib.readthedocs.io/en/latest/component/workflow.html",
    },
    "qlib_native_api": qlib_workflow_api,
    "sidecars": {
        "evolution_summary": "evolution_summary.json",
        "selected_features": "selected_features.csv",
        "failure_modes": "failure_modes.md",
    },
    "recorder_artifacts": recorder_artifacts,
    "artifact_policy": {
        "policy": "slim_card_recorder_first",
        "workflow_config": "workflow.yaml",
        "primary_artifact": f"{EXPERIMENT}.ipynb",
        "machine_entrypoint": "summary.json",
        "kept_sidecars": ["evolution_summary.json", "failure_modes.md", "selected_features.csv"],
        "recorder_only_artifacts": {
            "candidate_pool": "artifacts/cognitive_alpha_evolution/candidate_pool/generation_*.jsonl",
            "fitness": "artifacts/cognitive_alpha_evolution/fitness/generation_*.csv",
            "lineage": "artifacts/cognitive_alpha_evolution/lineage/generation_*.jsonl",
            "quality": "artifacts/cognitive_alpha_evolution/quality/generation_*.csv",
            "rejected": "artifacts/cognitive_alpha_evolution/rejected/generation_*.jsonl",
            "state": "artifacts/cognitive_alpha_evolution/state.json",
        },
    },
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "conclusion.md").write_text(
    "# 结论\n\n"
    f"- what_ran: 执行 {EXPERIMENT}，用 qlib.init、D.features、DatasetH/DataHandlerLP、Alpha158.get_feature_config 和 R.start 跑 soxl_full CogAlpha evolution。\n"
    f"- key_metrics: profile=soxl_full，generations={profile.generations}，initial_pool_size={profile.initial_pool_size}，parent_pool_size={profile.parent_pool_size}，children_per_generation={profile.children_per_generation}，selected_count={evolution.summary['selected_count']}，candidate_metric_rows={evolution.summary['candidate_metric_rows']}。\n"
    f"- artifact: qlib recorder id={recorder_id}，candidate/fitness/quality/lineage/rejected/state 全部保存在 recorder。\n"
    "- promotion_decision: not_promoted_full_evolution_only。full evolution 只证明可恢复候选进化链路，不单独晋升生产。\n"
    "- risks: 该 full run 是 repo-owned deterministic Code/Alpha158 生成，不依赖外部 LLM provider；下游仍需重新接 LightGBM 仓位模型和 benchmark。\n"
    "- next_action: 用 full selected features 重跑 position model，并接入 signal refresh/Bark paper-sim push。\n"
    "- 本研究面向交易复盘和风险控制，不构成买卖建议。\n",
    encoding="utf-8",
)
print(json.dumps({"summary": "summary.json", "recorder_artifacts": len(recorder_artifacts)}, indent=2))

{
  "summary": "summary.json",
  "recorder_artifacts": 122
}


## 中文结论

见 `conclusion.md`。本研究面向交易复盘和风险控制，不构成买卖建议。